# Chuẩn bị và kiểm tra dữ liệu field/layout hóa đơn

Notebook clone repo public, cố định revision bằng SHA tại đầu lần chạy, xác thực schema/checksum/split rồi xuất báo cáo và ZIP vào Google Drive. Không huấn luyện model.

In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
import json, shutil, subprocess, sys

drive.mount('/content/drive')
run_id = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
drive_run = Path('/content/drive/MyDrive/FieldLayoutLabeler/runs') / run_id
drive_run.mkdir(parents=True, exist_ok=False)
repo_url = 'https://github.com/nguyenhung1204/data_detection-layout.git'
repo = Path('/content/data_detection_layout')
if repo.exists(): shutil.rmtree(repo)
subprocess.run(['git', 'clone', '--single-branch', '--branch', 'main', repo_url, str(repo)], check=True)
commit_sha = subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip()
subprocess.run(['git', '-C', str(repo), 'checkout', '--detach', commit_sha], check=True)
print('Đang kiểm tra commit:', commit_sha)


In [ ]:
%pip -q install -r /content/data_detection_layout/requirements.txt
sys.path.insert(0, str(repo))
from tools.validate_dataset import validate
from tools.export_dataset import export_dataset

validation = validate(repo)
metadata = {
    'run_id_utc': run_id,
    'repository': repo_url,
    'commit_sha': commit_sha,
    'python': sys.version,
    'validation': validation,
    'fine_tuning_performed': False,
}
(drive_run / 'metadata.json').write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8')
(drive_run / 'validation.json').write_text(json.dumps(validation, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(validation, ensure_ascii=False, indent=2))
if not validation['valid']:
    raise RuntimeError('Validation có lỗi. Xem validation.json trên Drive; không tạo gói export.')


In [ ]:
export_path = drive_run / 'field_layout_export.zip'
export_dataset(repo, export_path)
manifest = repo / 'data/manifests/invoice_manifest.csv'
shutil.copy2(manifest, drive_run / 'invoice_manifest.csv')
summary = {
    'repository_commit': commit_sha,
    'invoice_count': validation['invoice_count'],
    'annotation_count': validation['annotation_count'],
    'split_invoice_counts': validation['split_invoice_counts'],
    'assignment_invoice_counts': validation['assignment_invoice_counts'],
    'annotation_status_counts': validation['annotation_status_counts'],
    'note': 'Generic field/layout annotation export only; not a model-specific PaddleOCR training package.',
}
(drive_run / 'SUMMARY.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
print('Kết quả đã lưu trên Drive:', drive_run)
print('ZIP:', export_path)
print(json.dumps(summary, ensure_ascii=False, indent=2))


## Ý nghĩa kết quả

`verified.jsonl` chỉ chứa nhãn field/layout đã xác nhận; `review.jsonl` giữ vùng OCR candidate hoặc nhãn chưa xác nhận. Recognition transcript đã rà không tự động xác nhận field/layout. ZIP này chưa phải bộ fine-tune PaddleOCR hoàn chỉnh vì model/config và định dạng training cho task detection/layout chưa được chọn trong phạm vi notebook.